# `paper_v4` Scope-3 retrain — Colab harness (any GPU)

Re-runs the v2 SPLM-family Tiny-Shakespeare experiments under the
v4 leak-free integrator (`cfg.causal_force = True`, the post-fix
default). All five cells share the v2 hyperparameters; the only
systematic change vs v2 is causal honesty in the SARF $\xi$
recomputation.

## Cells

| # | Cell                          | Trainer                                                           | Leak-immune? |
|---|-------------------------------|-------------------------------------------------------------------|:------------:|
| 1 | `splm_baseline`               | `train_splm.py`                                                   |     yes      |
| 2 | `splm_sarf`                   | `sarf_variant/train_splm_sarf.py`                                 |      no      |
| 3 | `splm_sarfmass_embed_head`    | `sarf_mass_variant/train_splm_sarf_mass.py --mass-mode embed_head` |      no      |
| 4 | `splm_sarfmass_logfreq`       | `sarf_mass_variant/train_splm_sarf_mass.py --mass-mode logfreq`   |      no      |
| 5 | `matched_baseline`            | `train_matched.py`                                                |     yes      |

Cells 1 and 5 are **leak-immune by construction** (no SARF $\xi$
recomputation; for the matched baseline, no SPLM integrator at all);
they serve as sanity-check controls. Cells 2–4 are the actual
Scope-3 retrains.

## Wall-clock estimates (production: 5 cells × 5 seeds = 25 runs)

| GPU                | per-seed (SPLM) | per-seed (matched) | total (25 runs) |
|--------------------|----------------:|-------------------:|----------------:|
| T4 (free Colab)    | ~10 min         | ~3 min             | **~3.5 h**      |
| L4 (Pro)           | ~7 min          | ~2 min             | **~2.5 h**      |
| A100 (Pro+)        | ~4 min          | ~1.5 min           | **~1.5 h**      |
| MPS (laptop, ref.) | ~22 min         | ~10 min            | ~9 h            |

All five cells fit on any Colab GPU (smallest model is ~7 M params,
Tiny Shakespeare is ~340 KB tokens, batch=16, block=128). The hard
Colab-session ceiling (12 h) is comfortably above even the T4
estimate.

## Numerics: TF32 forced off

On Ampere+ GPUs (L4/A100/H100) the default fp32 matmul path uses
**TF32** — a 19-bit format with only **10 mantissa bits** vs true
fp32's 23. The SPLM forward computes its force field via
`torch.autograd.grad(…, create_graph=True)`, which is the most
numerically sensitive operation in the whole pipeline; under TF32 we
previously observed PPL inflation in the H100 scale-up pilot. The
notebook therefore forces TF32 OFF in two redundant ways:

1. **Driver-level:** `NVIDIA_TF32_OVERRIDE=0` is set in the notebook
   environment before any subprocess is spawned, so cuBLAS / cuDNN
   refuse the TF32 path even if Python forgets to ask.
2. **PyTorch-level:** every trainer (`train_splm.py`,
   `sarf_variant/train_splm_sarf.py`,
   `sarf_mass_variant/train_splm_sarf_mass.py`,
   `train_matched.py`) now exposes a `--allow-tf32` flag that
   defaults to **False** and explicitly sets
   `torch.backends.cuda.matmul.allow_tf32 = False` and
   `torch.backends.cudnn.allow_tf32 = False` on entry.

T4 GPUs are pre-Ampere and have no TF32 path at all, so on the free
Colab tier this is a documented no-op. On L4/A100/H100 it costs a
~2x slowdown in the matmul kernels in exchange for clean fp32
second-order numerics (the time estimates above already include
this cost).

## Companion files

* [`SCOPE3_README.md`](SCOPE3_README.md) — full instructions and
  decision rules for `paper_v4`.
* [`multi_seed_runner.py`](multi_seed_runner.py) — the runner this
  notebook drives.
* [`scope3_comparison.py`](scope3_comparison.py) — Scope-3-specific
  v2-vs-v4 headline aggregator (called at the bottom).


## 1. Environment setup


In [ ]:
import os, sys, subprocess, shutil, json, time
from pathlib import Path

# Set CUDA allocator config BEFORE any torch import or CUDA context
# creation. expandable_segments=True helps with fragmentation; harmless
# on T4/L4 and recommended on A100/H100.
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

# Force TF32 OFF at the cuBLAS / cuDNN driver level for every Python
# subprocess that inherits this env. TF32 has only a 10-bit mantissa
# (vs 23 bits for true fp32) and the SPLM forward computes its force
# field via torch.autograd.grad(..., create_graph=True), which is the
# numerically most sensitive part of the pipeline. Belt-and-suspenders
# with the per-trainer torch APIs (the trainers also call
# torch.backends.cuda.matmul.allow_tf32 = False; pass --allow-tf32 to
# any trainer to opt back in for a precision-artifact reference run).
# T4 GPUs are pre-Ampere and have no TF32 path at all, so this is a
# no-op there; on L4/A100/H100 it forces real fp32 matmuls.
os.environ.setdefault('NVIDIA_TF32_OVERRIDE', '0')

# Decide where the repo will live. On Colab the workspace is /content.
REPO_PARENT = Path('/content') if Path('/content').exists() else Path.cwd()
print('REPO_PARENT =', REPO_PARENT)


In [ ]:
# Clone (or pull) the semsimula repo into Colab's ephemeral disk.
# Replace REPO_URL with your fork or branch as needed.
REPO_URL  = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_NAME = 'semsimula'
REPO_DIR  = REPO_PARENT / REPO_NAME

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)],
                   check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                   check=False)

MULTISEED_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'multi_seed'
print('REPO_DIR     :', REPO_DIR)
print('MULTISEED_DIR:', MULTISEED_DIR)


In [ ]:
# Install Python dependencies. Colab already ships with a recent
# torch+CUDA; we only need to top up the smaller helpers.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'datasets', 'huggingface_hub'], check=True)
print('Dependencies OK')


In [ ]:
# Verify GPU + report device specs and TF32 state.
import torch
print('torch      :', torch.__version__)
print('cuda avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device     :', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'memory     : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')
    cap = torch.cuda.get_device_capability(0)
    is_ampere_plus = cap[0] >= 8
    print(f'capability : sm_{cap[0]}{cap[1]}  '
          f'(Ampere+ -> TF32 path exists: {is_ampere_plus})')
    # Belt-and-suspenders: also disable TF32 in this notebook process
    # (the env var covers the subprocesses; this covers anything we run
    # inline in the notebook itself).
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 (env) : NVIDIA_TF32_OVERRIDE =',
          os.environ.get('NVIDIA_TF32_OVERRIDE'))
    print('TF32 matmul:', torch.backends.cuda.matmul.allow_tf32,
          '(False = forcing real fp32, 23-bit mantissa)')
    print('TF32 cudnn :', torch.backends.cudnn.allow_tf32,
          '(False = forcing real fp32, 23-bit mantissa)')
else:
    print('!!! CUDA not available — this notebook requires GPU.')
    print('!!! Runtime → Change runtime type → GPU (T4/L4/A100 all work).')


In [ ]:
# Verify data + logfreq surprisal files are present (tracked in repo).
DATA_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'data'
SARF_MASS_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'sarf_mass_variant'
LOGFREQ_PATH = SARF_MASS_DIR / 'results' / 'logfreq_surprisal.npy'
SHAKESPEARE_NPZ = DATA_DIR / 'tinyshakespeare_gpt2.npz'

for p in (SHAKESPEARE_NPZ, LOGFREQ_PATH):
    assert p.exists(), f'Missing tracked artefact: {p}'
    print(f'OK  {p.relative_to(REPO_DIR)}  ({p.stat().st_size/1e3:.1f} KB)')


## 2. (Optional) Mount Google Drive for persistent results

If you want artefacts to survive Colab session disconnects, mount
Drive and set `DRIVE_RESULTS` below. Otherwise the runner writes
into the repo's `notebooks/conservative_arch/multi_seed/results/`
directory inside Colab's ephemeral disk, and you'll need to copy
the `scope3_shakespeare/` subdir somewhere persistent before the
session ends.

Skip this section if you intend to commit the results back to git
from inside the same session.


In [ ]:
# Uncomment to mount Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_scope3')
# DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
# print('DRIVE_ROOT:', DRIVE_ROOT)
DRIVE_ROOT = None
print('DRIVE_ROOT not configured (results will live in repo dir).')


## 3. Pre-flight smoke (~3 min)

Single seed × single cell at `--mode smoke` (300 steps, batch=8,
block=64). Validates the entire pipeline — model build, data load,
GPU availability, gradient flow, checkpoint write — without burning
Colab session time. **Always run this first** before launching the
production cells.


In [ ]:
RUNNER_PY = MULTISEED_DIR / 'multi_seed_runner.py'
AGGREGATOR_PY = MULTISEED_DIR / 'multi_seed_aggregator.py'
SCOPE3_PY = MULTISEED_DIR / 'scope3_comparison.py'

smoke_cmd = [
    sys.executable, str(RUNNER_PY),
    '--mode', 'smoke',
    '--n-seeds', '1',
    '--models', 'splm_baseline',
    '--tag', 'scope3_smoke',
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
]
print('CMD:', ' '.join(smoke_cmd))
rc = subprocess.run(smoke_cmd, cwd=str(REPO_DIR)).returncode
assert rc == 0, f'smoke run returned non-zero exit code: {rc}'
print('\nSmoke run OK.')


In [ ]:
# Inspect the smoke run output.
smoke_dir = MULTISEED_DIR / 'results' / 'scope3_smoke' / 'splm_baseline' / 'seed_0'
print('Smoke artefacts:')
for p in sorted(smoke_dir.glob('*')):
    print(' ', p.name, f'({p.stat().st_size:,} bytes)')
summary = next(smoke_dir.glob('*_summary.md'), None)
if summary is not None:
    print('\n--- summary head ---')
    print('\n'.join(summary.read_text().splitlines()[:25]))


## 4. Production retrain — 5 cells × 5 seeds

Two cells, one per phase:

* **Phase A — 4-cell mass ablation** (`splm_baseline`, `splm_sarf`,
  `splm_sarfmass_embed_head`, `splm_sarfmass_logfreq`).
* **Phase B — matched-baseline pairing** (`matched_baseline`).

Phase B is split out so the matched baseline (which has no SPLM
integrator and is ~3× faster per seed) can be launched in parallel
from a second Colab session if you want to compress wall-clock
further. By default they run sequentially in the same session.

Both phases use `--skip-existing`: any seed whose checkpoint
already lives in the destination directory is skipped, so you can
re-launch after a Colab disconnect without losing progress.


In [ ]:
# Configure the production run.
TAG = 'scope3_shakespeare'
SEEDS = [0, 1, 2, 3, 4]                     # set to [0] for an S=1 dry run
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PHASE_A_MODELS = (
    'splm_baseline,splm_sarf,'
    'splm_sarfmass_embed_head,splm_sarfmass_logfreq'
)
PHASE_B_MODELS = 'matched_baseline'

RUN_ROOT = MULTISEED_DIR / 'results' / TAG
print(f'TAG     = {TAG}')
print(f'SEEDS   = {SEEDS}')
print(f'DEVICE  = {DEVICE}')
print(f'RUN_ROOT= {RUN_ROOT}')


In [ ]:
# Phase A — 4-cell mass ablation (4 cells × len(SEEDS) seeds).
# Expect ~3 h on T4, ~2 h on L4, ~80 min on A100 for SEEDS=[0..4].
phase_a_cmd = [
    sys.executable, str(RUNNER_PY),
    '--mode', 'shakespeare',
    '--seeds', ','.join(str(s) for s in SEEDS),
    '--models', PHASE_A_MODELS,
    '--tag', TAG,
    '--device', DEVICE,
    '--skip-existing',
]
print('CMD:', ' '.join(phase_a_cmd))
t0 = time.time()
rc = subprocess.run(phase_a_cmd, cwd=str(REPO_DIR)).returncode
dt = time.time() - t0
print(f'\nPhase A done in {dt/60:.1f} min, rc={rc}')
assert rc == 0, f'phase A returned non-zero exit code: {rc}'


In [ ]:
# Phase B — matched baseline (1 cell × len(SEEDS) seeds).
# Expect ~25 min on T4, ~15 min on L4, ~10 min on A100 for SEEDS=[0..4].
phase_b_cmd = [
    sys.executable, str(RUNNER_PY),
    '--mode', 'shakespeare',
    '--seeds', ','.join(str(s) for s in SEEDS),
    '--models', PHASE_B_MODELS,
    '--tag', TAG,
    '--device', DEVICE,
    '--skip-existing',
]
print('CMD:', ' '.join(phase_b_cmd))
t0 = time.time()
rc = subprocess.run(phase_b_cmd, cwd=str(REPO_DIR)).returncode
dt = time.time() - t0
print(f'\nPhase B done in {dt/60:.1f} min, rc={rc}')
assert rc == 0, f'phase B returned non-zero exit code: {rc}'


## 5. Aggregation

Two passes:

1. **`multi_seed_aggregator.py`** — per-cell mean ± std + Welch's
   pairwise table on final val PPL + multi-seed loss-curve overlays.
2. **`scope3_comparison.py`** — Scope-3-specific narrative: v2 buggy
   vs v4 leak-free PPL, asymmetric inflation factor per cell,
   qualitative-direction preservation check, and four explicit
   decision rules for `paper_v4`.


In [ ]:
# Pass 1: standard multi-seed aggregator.
agg_cmd = [
    sys.executable, str(AGGREGATOR_PY),
    '--tag', TAG,
]
print('CMD:', ' '.join(agg_cmd))
rc = subprocess.run(agg_cmd, cwd=str(REPO_DIR)).returncode
print(f'Pass 1 (standard aggregator) rc={rc}')


In [ ]:
# Pass 2: Scope-3 v2-vs-v4 comparison.
scope3_cmd = [
    sys.executable, str(SCOPE3_PY),
    '--tag', TAG,
]
print('CMD:', ' '.join(scope3_cmd))
rc = subprocess.run(scope3_cmd, cwd=str(REPO_DIR)).returncode
print(f'Pass 2 (scope3_comparison) rc={rc}')


In [ ]:
# Display the headline scope3_comparison.md report.
from IPython.display import Markdown, display
report = RUN_ROOT / 'scope3_comparison.md'
if report.exists():
    display(Markdown(report.read_text()))
else:
    print('!!! report missing:', report)


In [ ]:
# Display the per-cell loss-curve overlays produced by Pass 1.
from IPython.display import Image, display
for png in sorted(RUN_ROOT.glob(f'{TAG}_loss_curves_*.png')):
    print(png.name)
    display(Image(filename=str(png)))


## 6. Persistence & next steps

All artefacts now live at `RUN_ROOT`:

```
notebooks/conservative_arch/multi_seed/results/scope3_shakespeare/
├── splm_baseline/             seed_0..4/
├── splm_sarf/                  seed_0..4/
├── splm_sarfmass_embed_head/   seed_0..4/
├── splm_sarfmass_logfreq/      seed_0..4/
├── matched_baseline/           seed_0..4/
├── run_log.jsonl                            # per-(model,seed) launch log
├── scope3_shakespeare_report.md             # standard multi-seed report
├── scope3_shakespeare_loss_curves_*.png     # per-cell overlay plots
└── scope3_comparison.md                     # v2-vs-v4 headline table
```

### Recommended persistence options

1. **rsync to Drive** if you mounted it in §2:
   ```python
   import shutil
   shutil.copytree(RUN_ROOT, DRIVE_ROOT / TAG, dirs_exist_ok=True)
   ```
2. **Push to git** (recommended for the small reports):
   ```bash
   cd $REPO_DIR
   git add notebooks/conservative_arch/multi_seed/results/scope3_shakespeare/
   git commit -m 'Scope-3 leak-free retrain (5 cells × 5 seeds, Colab GPU)'
   git push
   ```
   Note: `*.pt` checkpoints and `*.trajectories.pkl` files are
   `.gitignore`d — only the JSONL logs, summary markdowns,
   loss-curve PNGs, and the two top-level reports are committed.

### Wiring the results into `paper_v4`

Open `paper_v4/sections/15_conservative_architectures.tex` and
follow the four decision rules at the bottom of
`scope3_comparison.md`:

1. Verify the leak-immunity controls (`splm_baseline`, `matched_baseline`).
2. Replace the `~2× asymmetric inflation` placeholder with the
   actual per-cell inflation factors.
3. Retire the `v2-historical` caveat blocks if the qualitative
   ordering survives.
4. Update the matched-baseline pairing in §15.

After updating §15, recompile `paper_v4/main.tex` and you are ready
for the freeze + Zenodo snapshot.
